# OSS MLOps workshop — Kubeflow + Feast (CPU-only)

Run cells **in order**. Edit **only** the configuration cell below (`WORKSHOP_NAMESPACE`).

Prerequisites: `oc` or `kubectl` authenticated; `feast` CLI installed in this environment; cluster has `ClusterTrainingRuntime` **torch-distributed**.

In [ ]:
from pathlib import Path

# --- attendee: set your namespace ---
WORKSHOP_NAMESPACE = "change-me-namespace"

# Workshop root = parent of notebooks/
WORKSHOP_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
print("WORKSHOP_ROOT:", WORKSHOP_ROOT)
print("WORKSHOP_NAMESPACE:", WORKSHOP_NAMESPACE)

## 1) Feast — build parquet and `feast apply`

Feature definitions expect `data/transactions.parquet` (we generate it from the bundled CSV).

In [ ]:
import shutil
import subprocess
import sys

import pandas as pd

feast_repo = WORKSHOP_ROOT / "feast_repo"
csv_path = feast_repo / "data" / "transactions.csv"
parquet_path = feast_repo / "data" / "transactions.parquet"

df = pd.read_csv(csv_path)
df["amount"] = df["amount"].astype("float32")
df["is_fraud"] = df["is_fraud"].astype("float32")
df.to_parquet(parquet_path, index=False)
print("Wrote", parquet_path)

if not shutil.which("feast"):
    print("Install Feast CLI in this image, e.g. pip install 'feast[aws]' or your offline wheel", file=sys.stderr)
else:
    subprocess.run(["feast", "apply"], cwd=feast_repo, check=True)

## 2) Trainer v2 — ConfigMap + `TrainJob` (CPU, `torch-distributed`)

We still **reference** the cluster `torch-distributed` runtime; **CPU** image and training files are supplied via `trainer` + `podTemplateOverrides`.

In [ ]:
import subprocess

cli = "oc"  # use "kubectl" if you prefer

def apply_manifest(name: str) -> None:
    raw = (WORKSHOP_ROOT / "manifests" / name).read_text()
    raw = raw.replace("REPLACE_NAMESPACE", WORKSHOP_NAMESPACE)
    subprocess.run([cli, "apply", "-f", "-", "-n", WORKSHOP_NAMESPACE], input=raw.encode(), check=True)

apply_manifest("workshop-training-configmap.yaml")
apply_manifest("trainjob-fraud-workshop.yaml")
print("Applied ConfigMap + TrainJob")

In [ ]:
import subprocess

cli = "oc"
subprocess.run(
    [cli, "get", "trainjob", "-n", WORKSHOP_NAMESPACE],
    check=False,
)
# When READY/Succeeded, optionally copy the model out of the train pod:
# oc get pods -n $NS -l trainer.kubeflow.org/trainjob=fraud-workshop-train
# oc cp <pod>:/workspace/out/model.pt ./model.pt

## 3) Kubeflow Pipelines — compile (and submit if client configured)

Compiles `pipeline/fraud_workshop_pipeline.py` to YAML next to it. Submit via UI or `kfp` client using your cluster’s KFP endpoint (facilitator documents URL in runbook).

In [ ]:
import subprocess
import sys

pipe = WORKSHOP_ROOT / "pipeline" / "fraud_workshop_pipeline.py"
subprocess.run([sys.executable, str(pipe)], cwd=pipe.parent, check=True)
print("Compiled pipeline YAML next to .py (if kfp SDK installed)")

In [ ]:
# Optional: submit with kfp (uncomment and set host)
# from kfp.client import Client
# client = Client(host="https://your-kfp-endpoint")
# client.create_run_from_pipeline_package(
#     pipeline_file=str(WORKSHOP_ROOT / "pipeline" / "fraud_workshop_pipeline.yaml"),
#     arguments={},
#     experiment_name="fraud-workshop",
#     namespace=WORKSHOP_NAMESPACE,
# )